# Homework 3 — Classification (ML Zoomcamp 2026)

Dataset: 2026 lead-scoring dataset. Target: `converted`.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mutual_info_score, accuracy_score
import sklearn
print(pd.__version__, np.__version__, sklearn.__version__)

3.0.6 2.5.3 1.9.1


In [2]:
df = pd.read_csv('data/course_lead_scoring_2026.csv')
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


## Data preparation

In [3]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [4]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [5]:
target = 'converted'
numerical = [c for c in df.select_dtypes(include='number').columns if c != target]
categorical = [c for c in df.columns if c not in numerical and c != target]
print('numerical:', numerical)
print('categorical:', categorical)

df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)
df.isnull().sum()

numerical: ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']
categorical: ['lead_source', 'industry', 'employment_status', 'location']


lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

## Question 1 — mode of `industry`

In [6]:
df['industry'].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [7]:
q1 = df['industry'].mode()[0]
print('Q1 answer:', q1)

Q1 answer: technology


## Question 2 — correlation matrix

In [8]:
corr = df[numerical].corr()
corr

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [9]:
pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count'),
]
pair_corr = {p: corr.loc[p[0], p[1]] for p in pairs}
for p, v in pair_corr.items():
    print(f'{p[0]:>25} & {p[1]:<20} {v:.4f}')
q2 = max(pair_corr, key=lambda p: abs(pair_corr[p]))
print('Q2 answer:', q2)

        interaction_count & lead_score           0.9157
 number_of_courses_viewed & lead_score           0.7572
 number_of_courses_viewed & interaction_count    0.7216
            annual_income & interaction_count    0.1228
Q2 answer: ('interaction_count', 'lead_score')


## Split the data

In [10]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train[target].values
y_val = df_val[target].values
y_test = df_test[target].values

del df_train[target]
del df_val[target]
del df_test[target]

len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

## Question 3 — mutual information (training set)

In [11]:
mi = {c: round(mutual_info_score(y_train, df_train[c]), 2) for c in categorical}
mi_raw = {c: mutual_info_score(y_train, df_train[c]) for c in categorical}
print(pd.DataFrame({'mi_rounded': mi, 'mi_raw': mi_raw}).sort_values('mi_raw', ascending=False))
q3 = max(mi_raw, key=mi_raw.get)
print('Q3 answer:', q3)

                   mi_rounded    mi_raw
lead_source              0.03  0.034346
employment_status        0.02  0.018966
industry                 0.00  0.001995
location                 0.00  0.001043
Q3 answer: lead_source


## Question 4 — logistic regression

In [12]:
def train_eval(features, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
    X_val = dv.transform(df_val[features].to_dict(orient='records'))
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    return accuracy_score(y_val, model.predict(X_val))

all_features = categorical + numerical
base_acc = train_eval(all_features)
print('accuracy:', base_acc)
print('Q4 answer:', round(base_acc, 2))

accuracy: 0.645
Q4 answer: 0.65


## Question 5 — feature elimination

In [13]:
rows = []
for f in all_features:
    acc = train_eval([x for x in all_features if x != f])
    rows.append({'feature': f, 'accuracy': acc, 'diff': base_acc - acc})
fe = pd.DataFrame(rows).sort_values('diff')
print(fe.to_string(index=False))

candidates = ['lead_source', 'number_of_courses_viewed', 'interaction_count']
cand = fe[fe.feature.isin(candidates)].set_index('feature')['diff']
q5 = cand.idxmin()
print('Q5 answer:', q5)

                 feature  accuracy   diff
           annual_income     0.724 -0.079
                industry     0.645  0.000
                location     0.645  0.000
       employment_status     0.645  0.000
              lead_score     0.644  0.001
number_of_courses_viewed     0.643  0.002
             lead_source     0.642  0.003
       interaction_count     0.601  0.044
Q5 answer: number_of_courses_viewed


## Question 6 — regularization

In [14]:
scores = {}
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    scores[C] = round(train_eval(all_features, C=C), 3)
    print(f'C={C:<10} accuracy={scores[C]}')
best = max(scores.values())
q6 = min(C for C, s in scores.items() if s == best)
print('Q6 answer:', q6)

C=1e-06      accuracy=0.598


C=1e-05      accuracy=0.598


C=0.0001     accuracy=0.613


C=0.001      accuracy=0.645
Q6 answer: 0.001


## Summary

In [15]:
print(f'Q1: {q1}')
print(f'Q2: {q2[0]} and {q2[1]}')
print(f'Q3: {q3}')
print(f'Q4: {round(base_acc, 2)}')
print(f'Q5: {q5}')
print(f'Q6: {q6}')

Q1: technology
Q2: interaction_count and lead_score
Q3: lead_source
Q4: 0.65
Q5: number_of_courses_viewed
Q6: 0.001
